# Docking Bench on Google Colab

Runs the Streamlit docking app on a Colab machine and prints a public
`https://....trycloudflare.com` link. Anyone with the link **and** the password
can sign in and dock.

**Run all cells** (Runtime > Run all). Keep this tab open: the link dies when
the Colab session ends (about 90 min idle, 12 h max on the free tier), and a
new run gives a new link. Results live on the Colab machine only - download
them with the last cell before it shuts down.

The repo is private: add a GitHub token (read access to `daga-pankaj/Dock-Bench`)
as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) and
allow this notebook to read it.

For gnina with CNN scoring on a GPU, pick Runtime > Change runtime type > T4 GPU
and tick `install_gnina` below.

In [ ]:
#@title Settings
password = ""  #@param {type:"string"}
user = "colleague"  #@param {type:"string"}
branch = "remodel"  #@param {type:"string"}
max_ligands = 20  #@param {type:"integer"}
install_gnina = False  #@param {type:"boolean"}

import secrets
if not password:
    password = secrets.token_urlsafe(9)
    print("No password set - generated one:", password)

In [ ]:
#@title Install (smina, Python packages, app)
import os, subprocess
from pathlib import Path
from google.colab import userdata

REPO = Path("/content/Dock-Bench")
token = userdata.get("GITHUB_TOKEN")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--branch", branch,
                    f"https://{token}@github.com/daga-pankaj/Dock-Bench.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)

# smina + obrms: apt first, conda-forge (micromamba) into .env-tools as fallback
apt = subprocess.run("apt-get -qq update && apt-get -qq install -y smina openbabel",
                     shell=True, capture_output=True, text=True)
if apt.returncode != 0 or not Path("/usr/bin/smina").exists():
    print("apt smina unavailable - installing from conda-forge")
    subprocess.run(
        "mkdir -p /content/.bin && curl -Ls https://micro.mamba.run/api/micromamba/linux-64/latest"
        " | tar -xj -C /content/.bin --strip-components=1 bin/micromamba && "
        f"/content/.bin/micromamba create -y -q -p {REPO}/.env-tools -c conda-forge smina openbabel",
        shell=True, check=True)

if install_gnina:
    gnina = REPO / ".env-tools" / "bin" / "gnina"
    gnina.parent.mkdir(parents=True, exist_ok=True)
    if not gnina.exists():
        print("downloading gnina (~1 GB)")
        subprocess.run(["curl", "-Lfs", "-o", str(gnina),
                        "https://github.com/gnina/gnina/releases/latest/download/gnina.cuda12.8.static"], check=True)
        gnina.chmod(0o755)

subprocess.run(["pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
print("installed")

In [ ]:
#@title Start the app and print the link
import re, time

env = dict(os.environ,
           DOCKING_AUTH_USER=user,
           DOCKING_AUTH_PASS=password,
           DOCKING_MAX_LIGANDS=str(max_ligands))
subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
app = subprocess.Popen(
    ["python", "-m", "streamlit", "run", "streamlit_app/app.py",
     "--server.port", "8501", "--server.address", "127.0.0.1"],
    cwd=REPO, env=env, stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)

if not Path("/content/.bin/cloudflared").exists():
    subprocess.run("mkdir -p /content/.bin && curl -Lfs -o /content/.bin/cloudflared "
                   "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
                   " && chmod +x /content/.bin/cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(
    ["/content/.bin/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8501"],
    stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", Path("/content/cloudflared.log").read_text())
    if m:
        url = m.group(0)
        break
if not url:
    raise RuntimeError("no tunnel URL - see /content/cloudflared.log")
print(f"Link:     {url}")
print(f"User:     {user}")
print(f"Password: {password}")
print("Keep this tab open. App log: /content/streamlit.log")

In [ ]:
#@title Download all results (zip of runs/)
import shutil
from google.colab import files
files.download(shutil.make_archive("/content/docking_runs", "zip", REPO / "runs"))